## Aqui foi feito tanto a conversão quanto as análises exploratórias

### Bibliotecas usadas para fazer as conversões

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score

###  1 - fazer a conversão dos dados que estavam em formato xlsx para o formato csv para que fosse possivel trabalhar com os dados

In [ ]:

df = pd.read_excel("taxas_rend/tx_rend_brasil_regioes_ufs_2023.xlsx", sheet_name=0, engine='openpyxl',skiprows=3)
df.columns = df.iloc[0]
df.columns = df.columns.astype(str)
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
df = df[1:].reset_index(drop=True)  
df.fillna(method="ffill", inplace=True)
df.to_csv("tx_rend_2023.csv", index=False, encoding="utf-8")


C:\Users\valderez\AppData\Local\Temp\ipykernel_41628\285695147.py:12: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method="ffill", inplace=True)


### 2 - Após os arquivos serem convertidos foi realizada a filtragem do valores desejados

In [ ]:

df = pd.read_csv('tx_rend_2008.csv')


df_filtrado = df[(df['Localização'] == 'Total') & (df['Dependência Administrativa'] == 'Total')]


df_resultado = df_filtrado[['Unidade Geográfica', 'Total_Abandono_EF2008', 'Total_Abandono_EM2008']]
df_resultado.columns = ['Estado', 'Taxa_Abandono_EF2008', 'Taxa_Abandono_EM2008']
df_resultado.to_csv('taxas_abandono_por_estado2008.csv', index=False, encoding='utf-8-sig')
df_municipios = pd.read_csv('alunos_matriculados/matriculas_fundamental_medio_2008.csv') 

df_estados = df_municipios.groupby('NO_UF', as_index=False).agg({
    'Total_Fundamental': 'sum',
    'Total_Medio': 'sum'
})

df_estados['Total_Geral'] = df_estados['Total_Fundamental'] + df_estados['Total_Medio']

df_estados.to_csv('matriculas_por_estado2008.csv', index=False, encoding='utf-8-sig')

#----------------------------------------------------------------------------------------------------------------

df_matriculas = pd.read_csv('matriculas_por_estado2008.csv')  #
df_taxas = pd.read_csv('taxas_abandono_por_estado2008.csv')   

df_taxas = df_taxas.rename(columns={'Estado': 'NO_UF'})

df_final = pd.merge(df_matriculas, df_taxas, on='NO_UF', how='left')

df_final['Abandono_EF'] = (df_final['Total_Fundamental'] * df_final['Taxa_Abandono_EF2008']) / 100
df_final['Abandono_EM'] = (df_final['Total_Medio'] * df_final['Taxa_Abandono_EM2008']) / 100

df_final['Abandono_EF'] = df_final['Abandono_EF'].round().fillna(0).astype(int)
df_final['Abandono_EM'] = df_final['Abandono_EM'].round().fillna(0).astype(int)

df_final['Total_Abandonos'] = df_final['Abandono_EF'] + df_final['Abandono_EM']

df_final = df_final.rename(columns={'NO_UF': 'Estado'})

df_final.to_csv('abandono_escolar_por_estado2008.csv', index=False, encoding='utf-8-sig')

Arquivo 'matriculas_por_estado.csv' criado com sucesso!

Exemplo das primeiras linhas:
      NO_UF  Total_Fundamental  Total_Medio  Total_Geral
0      Acre           164364.0      33113.0     197477.0
1   Alagoas           660805.0     128937.0     789742.0
2     Amapá           142182.0      35733.0     177915.0
3  Amazonas           796224.0     159656.0     955880.0
4     Bahia          2622197.0     650127.0    3272324.0

Arquivo 'abandono_escolar_por_estado.csv' gerado com sucesso!

Resumo estatístico:
       Abandono_EF  Abandono_EM  Total_Abandonos
count         27.0         27.0             27.0
mean           0.0          0.0              0.0
std            0.0          0.0              0.0
min            0.0          0.0              0.0
25%            0.0          0.0              0.0
50%            0.0          0.0              0.0
75%            0.0          0.0              0.0
max            0.0          0.0              0.0

5 primeiros registros:
     Estado  Total_Fun

 ### 3 - Nesse etapa foi feito o cálculo do valor total dos abandonos de cada ano com base nas taxas de abandono e da quantidade total de alunos. O código foi ajustado e reutilizado para outros anos

In [ ]:
df_matriculas = pd.read_csv('alunos_matriculados/matriculas_por_estado2010.csv')  
df_taxas = pd.read_csv('rendimento_escolar/Dados Tratados/taxas_abandono_por_estado2010.csv')   


df_taxas = df_taxas.rename(columns={'Estado': 'NO_UF'})


df_final = pd.merge(df_matriculas, df_taxas, on='NO_UF', how='left')


df_final['Abandono_EF'] = (df_final['Total_Fundamental'] * df_final['Taxa_Abandono_EF2010']) / 100
df_final['Abandono_EM'] = (df_final['Total_Medio'] * df_final['Taxa_Abandono_EM2010']) / 100


df_final['Abandono_EF'] = df_final['Abandono_EF'].round().fillna(0).astype(int)
df_final['Abandono_EM'] = df_final['Abandono_EM'].round().fillna(0).astype(int)

df_final['Total_Abandonos'] = df_final['Abandono_EF'] + df_final['Abandono_EM']

df_final = df_final.rename(columns={'NO_UF': 'Estado'})

df_final.to_csv('abandono_escolar_por_estado2010.csv', index=False, encoding='utf-8-sig')


Arquivo 'abandono_escolar_por_estado.csv' gerado com sucesso!

Resumo estatístico:
        Abandono_EF    Abandono_EM  Total_Abandonos
count     27.000000      27.000000        27.000000
mean   19874.851852   38765.333333     58640.185185
std    21529.108240   32203.625603     50129.249166
min     1099.000000    1514.000000      2613.000000
25%     4485.500000   12649.500000     17669.000000
50%    13949.000000   32148.000000     50595.000000
75%    27131.500000   55263.000000     83467.000000
max    80850.000000  114268.000000    195118.000000

5 primeiros registros:
     Estado  Total_Fundamental  Total_Medio  Total_Geral  \
0      Acre           166200.0      36295.0     202495.0   
1   Alagoas           630889.0     130247.0     761136.0   
2     Amapá           144597.0      37871.0     182468.0   
3  Amazonas           771963.0     162113.0     934076.0   
4     Bahia          2450008.0     589012.0    3039020.0   

   Taxa_Abandono_EF2010  Taxa_Abandono_EM2010  Abandono_EF  Aba

tratamento dos alores das variaveis para que tenham apenas 2 casas decimais

### 4 - Convertendo dados do bolsa família em csv

#### baixando os dados do bolsa familias e convertendo para o formato CSV

In [ ]:


url = "https://aplicacoes.mds.gov.br/sagi/servicos/misocial/?fq=anomes_s:2023*&fl=codigo_ibge%2Canomes_s%2Cqtd_familias_beneficiarias_bolsa_familia_s%2Cvalor_repassado_bolsa_familia_s%2Cpbf_vlr_medio_benef_f&fq=valor_repassado_bolsa_familia_s%3A*&q=*%3A*&rows=100000&sort=anomes_s%20desc%2C%20codigo_ibge%20asc&wt=csv"

dados_bf = pd.read_csv(url, encoding='latin1', sep=';')

dados_bf.to_csv('valorRepassado_familia_2023.csv', index=False, encoding='utf-8-sig')

print("Arquivo salvo com sucesso!")

dados = pd.read_csv('valorRepassado_familia_2023.csv', sep=';', encoding='latin1')

dados = dados.replace({'"': ''}, regex=True)

dados.to_csv('valorRepassado_familia_2023.csv', index=False, sep=';', encoding='utf-8-sig')

print("Arquivo corrigido salvo sem aspas!")


Arquivo salvo com sucesso!
Arquivo corrigido salvo sem aspas!


In [ ]:
uf_map = {
    11: {'sigla': 'RO', 'nome': 'Rondônia'},
    12: {'sigla': 'AC', 'nome': 'Acre'},
    13: {'sigla': 'AM', 'nome': 'Amazonas'},
    14: {'sigla': 'RR', 'nome': 'Roraima'},
    15: {'sigla': 'PA', 'nome': 'Pará'},
    16: {'sigla': 'AP', 'nome': 'Amapá'},
    17: {'sigla': 'TO', 'nome': 'Tocantins'},
    21: {'sigla': 'MA', 'nome': 'Maranhão'},
    22: {'sigla': 'PI', 'nome': 'Piauí'},
    23: {'sigla': 'CE', 'nome': 'Ceará'},
    24: {'sigla': 'RN', 'nome': 'Rio Grande do Norte'},
    25: {'sigla': 'PB', 'nome': 'Paraíba'},
    26: {'sigla': 'PE', 'nome': 'Pernambuco'},
    27: {'sigla': 'AL', 'nome': 'Alagoas'},
    28: {'sigla': 'SE', 'nome': 'Sergipe'},
    29: {'sigla': 'BA', 'nome': 'Bahia'},
    31: {'sigla': 'MG', 'nome': 'Minas Gerais'},
    32: {'sigla': 'ES', 'nome': 'Espírito Santo'},
    33: {'sigla': 'RJ', 'nome': 'Rio de Janeiro'},
    35: {'sigla': 'SP', 'nome': 'São Paulo'},
    41: {'sigla': 'PR', 'nome': 'Paraná'},
    42: {'sigla': 'SC', 'nome': 'Santa Catarina'},
    43: {'sigla': 'RS', 'nome': 'Rio Grande do Sul'},
    50: {'sigla': 'MS', 'nome': 'Mato Grosso do Sul'},
    51: {'sigla': 'MT', 'nome': 'Mato Grosso'},
    52: {'sigla': 'GO', 'nome': 'Goiás'},
    53: {'sigla': 'DF', 'nome': 'Distrito Federal'}
}

ano = 2021 

df = pd.read_csv("valorRepassado_familia_2021.csv")  

df['cod_uf'] = df['ibge'].astype(str).str[:2].astype(int)

df['uf_sigla'] = df['cod_uf'].map(lambda x: uf_map[x]['sigla'])
df['uf_nome'] = df['cod_uf'].map(lambda x: uf_map[x]['nome'])

df_estado = df.groupby(['uf_sigla', 'uf_nome']).agg({
    'valor_repassado_bolsa_familia': 'sum',
    'qtd_familias_beneficiarias_bolsa_familia': 'sum'
}).reset_index()

df_estado['valor_medio_bf'] = df_estado['valor_repassado_bolsa_familia'] / df_estado['qtd_familias_beneficiarias_bolsa_familia']
df_estado['ano'] = ano  

df_estado = df_estado.sort_values('uf_sigla')

resultado = df_estado[['ano', 'uf_sigla', 'uf_nome', 'qtd_familias_beneficiarias_bolsa_familia', 'valor_repassado_bolsa_familia', 'valor_medio_bf']]

nome_arquivo_saida = f'bolsa_familia_por_estado_{ano}.csv'
resultado.to_csv(nome_arquivo_saida, index=False, float_format='%.2f')

print(f"Arquivo salvo: {nome_arquivo_saida}")
print(resultado.head())

Arquivo salvo: bolsa_familia_por_estado_2021.csv
    ano uf_sigla   uf_nome  qtd_familias_beneficiarias_bolsa_familia  \
0  2021       AC      Acre                                  907936.0   
1  2021       AL   Alagoas                                 4132720.0   
2  2021       AM  Amazonas                                 4059479.0   
3  2021       AP     Amapá                                  758406.0   
4  2021       BA     Bahia                                18580643.0   

   valor_repassado_bolsa_familia  valor_medio_bf  
0                   1.844038e+08      203.102209  
1                   4.780583e+08      115.676420  
2                   6.279694e+08      154.692105  
3                   1.102775e+08      145.407009  
4                   2.060550e+09      110.897653  


### 5 - Adicionar a coluna contendo a sigla de cada estado para facilitar a manipulação posterior dos dados, converte os dados para que os valores de quantidade de familias e o valor medio da bolsa familia sejam agrupados por estados

In [ ]:
df = pd.read_csv('bolsa_familia_por_estado_2021.csv')  


if 'valor_medio_bf' in df.columns:

    df['valor_medio_bf_ajustado'] = (df['valor_medio_bf'] * 1.02985750).round(2)
    

    df.to_csv('bolsa_familia_por_estado_2021.csv', index=False, float_format='%.2f')
    
    print("Ajuste realizado com sucesso! Resultado com 2 casas decimais:")
    print(df[['uf_sigla', 'valor_medio_bf', 'valor_medio_bf_ajustado']].head())
else:
    print("Erro: Coluna 'valor_medio_bf' não encontrada.")
    print("Colunas disponíveis:", list(df.columns))

Ajuste realizado com sucesso! Resultado com 2 casas decimais:
  uf_sigla  valor_medio_bf  valor_medio_bf_ajustado
0       AC          203.10                   209.16
1       AL          115.68                   119.13
2       AM          154.69                   159.31
3       AP          145.41                   149.75
4       BA          110.90                   114.21


raliza o tratamento de dados dos dados relacionados ao desemprenho escolar 

In [ ]:
df2016 = pd.read_csv("rendimento_escolar/Dados Tratados/dadosTratados2020.csv")
print(df2016.columns.tolist())

#### filtragem as colunas do total Rural E Urbano

In [ ]:
df2023 = pd.read_csv("rendimento_escolar/Dados Tratados/dadosTratados2023.csv")
matriculas_df = pd.read_csv("alunos_matriculados/matriculas_fundamental_medio_2023.csv")

colunas_selecionadas = [
    'Ano', 'Região', 'UF', 'Código do Município', 'Nome do Município',
    'Localização', 'Rede', 'Total_Abandono_EF2023', 'Total_Abandono_EM2023'
]

dados_tratados_df = df2023[
    (df2023['Localização'] == 'Total') & 
    (df2023['Rede'] == 'Total')
][colunas_selecionadas].copy()

matriculas_df['CO_MUNICIPIO'] = matriculas_df['CO_MUNICIPIO'].astype(str).str.zfill(6)
dados_tratados_df['Código do Município'] = dados_tratados_df['Código do Município'].astype(str).str.zfill(6)

merged_df = pd.merge(
    matriculas_df,
    dados_tratados_df,
    left_on='CO_MUNICIPIO',
    right_on='Código do Município',
    how='left'
)

cols_numericas = ['Total_Fundamental', 'Total_Medio', 'Total_Abandono_EF2023', 'Total_Abandono_EM2023']
merged_df[cols_numericas] = merged_df[cols_numericas].apply(pd.to_numeric, errors='coerce')

merged_df['Abandono_Abs_EF'] = (merged_df['Total_Fundamental'] * merged_df['Total_Abandono_EF2023'] / 100).fillna(0)
merged_df['Abandono_Abs_EM'] = (merged_df['Total_Medio'] * merged_df['Total_Abandono_EM2023'] / 100).fillna(0)

agrupado_por_uf = merged_df.groupby('UF').agg({
    'Total_Fundamental': 'sum',
    'Total_Medio': 'sum',
    'Abandono_Abs_EF': 'sum',
    'Abandono_Abs_EM': 'sum',
    'Total_Abandono_EF2023': 'mean',
    'Total_Abandono_EM2023': 'mean'
}).reset_index()

agrupado_por_uf.to_csv("Estados2023_robusto.csv", sep=",", index=False)


#### Mescla os dois dataframes para incluir as colunas Total_Rural e Total_Urbano da planilha matriculas_detalhadas

In [ ]:

matriculas_df.columns = matriculas_df.columns.str.strip()
dados_tratados_df.columns = dados_tratados_df.columns.str.strip()

matriculas_df['CO_MUNICIPIO'] = matriculas_df['CO_MUNICIPIO'].astype(str)
dados_tratados_df['Código do Município'] = dados_tratados_df['Código do Município'].astype(str)


merged_df = pd.merge(matriculas_df, dados_tratados_df, left_on='CO_MUNICIPIO', right_on='Código do Município', how='outer')
merged_df[['Ano','CO_MUNICIPIO','Nome do Município','Localização','Total_Fundamental','Total_Medio', 'Total_Abandono_EF2008', 'Total_Abandono_EM2008']]


### 6 - Realiza o tratamento de dados por meio da multipliclação de alunos matriculados pela taxa de abandono para obter o número total de abandono tanto no ensino fundamental quanto no ensino medio e depois incluir esses valores no dataframe original

In [ ]:
df_matriculas = pd.read_csv('matriculas_por_estado2008.csv') 
df_taxas = pd.read_csv('taxas_abandono_por_estado2008.csv')   

df_taxas = df_taxas.rename(columns={'Estado': 'NO_UF'})

df_final = pd.merge(df_matriculas, df_taxas, on='NO_UF', how='left')


df_final['Abandono_EF'] = (df_final['Total_Fundamental'] * df_final['Taxa_Abandono_EF2008']) / 100
df_final['Abandono_EM'] = (df_final['Total_Medio'] * df_final['Taxa_Abandono_EM2008']) / 100
df_final['Abandono_EF'] = df_final['Abandono_EF'].round().fillna(0).astype(int)
df_final['Abandono_EM'] = df_final['Abandono_EM'].round().fillna(0).astype(int)


df_final['Total_Abandonos'] = df_final['Abandono_EF'] + df_final['Abandono_EM']

df_final = df_final.rename(columns={'NO_UF': 'Estado'})

df_final.to_csv('abandono_escolar_por_estado2008.csv', index=False, encoding='utf-8-sig')

#### Filtra os dados importantes, esse processo é repetido por cada ano 

In [ ]:
acress_df = merged_df[['UF', 'Resultado_Multiplicacao_EF2008', 'Resultado_Multiplicacao_EM2008']]
agrupado_por_uf = acress_df.groupby('UF', as_index=False).sum()
agrupado_por_uf

### Mesclar todos os dados do abandono escolar em apenas um dataframe 

In [ ]:
ano2023 = pd.read_csv("total_abandono_alunos_estados/abandono_escolar_por_estado2023.csv")

### recorta as colunas que são importantes

In [ ]:
novo2023= ano2023[['Estado','Abandono_EF','Abandono_EM']]

### mescla os dados em um dataframe unico

In [ ]:
merged_df = pd.merge(dados, novo2023, left_on='Estado', right_on='Estado', how='outer')

### salva esse dataframe que contem os dados de todos os anos como um novo arquivo

In [ ]:
totalTotal.to_csv("totais por ano.csv", sep=",", index=False)

### filtra as colunas que são importantes e as renomeia

In [ ]:
novobolsa2023= bolsa2023[['uf_nome','valor_medio_estadual','qtd_familias_beneficiarias_bolsa_familia_ajustado']]
novobolsa2023.rename(columns={'uf_nome':'Estado'}, inplace=True)

### salva em um dataframe os dados de todos os anos

In [ ]:
merged_df1 = pd.merge(dados1, novobolsa2023, left_on='Estado', right_on='Estado', how='outer')

In [ ]:
totalTotal= merged_df1

### salva esse dataframe como sendo um arquivo

In [ ]:
totalTotal.to_csv("totais por ano.csv", sep=",", index=False)

#### 7 - foram incluídas variáveis de taxa de Desocupação, nivel de Ocupação e taxa de participação. Para essas variáveis foi realizado um processo de tratamento para filtrar os dados importantes

In [ ]:
taxas = pd.read_csv("INDICADORES_TABELA2.csv")

df_segundo_trimestre = taxas[taxas['Trimestre'] == 2]
colunas_desejadas = ['UF', 'Ano', 'Trimestre', 'Taxa_Desocupacao', 'Nivel_Ocupacao', 'Taxa_Participacao']
df_filtrado = df_segundo_trimestre[colunas_desejadas]

df_filtrado = df_filtrado.reset_index(drop=True)

mapa_estados = {
    "Acre": "AC", "Alagoas": "AL", "Amapá": "AP", "Amazonas": "AM", "Bahia": "BA",
    "Ceará": "CE", "Espírito Santo": "ES", "Goiás": "GO", "Maranhão": "MA", "Mato Grosso": "MT",
    "Mato Grosso do Sul": "MS", "Minas Gerais": "MG", "Pará": "PA", "Paraíba": "PB", "Paraná": "PR",
    "Pernambuco": "PE", "Piauí": "PI", "Rio de Janeiro": "RJ", "Rio Grande do Norte": "RN", "Rio Grande do Sul": "RS",
    "Rondônia": "RO", "Roraima": "RR", "Santa Catarina": "SC", "São Paulo": "SP", "Sergipe": "SE",
    "Tocantins": "TO", "Distrito Federal": "DF"
}

df_filtrado['UF'] = df_filtrado['UF'].apply(lambda x: mapa_estados.get(x, x)) 

df_filtrado.to_csv("TaxasDeDesocupacaoOcupacaoParticipacao.csv",index=False)

### 8 - Depois de tar trarados todos os dados iniciou-se a etapa de mesclar todos os dados um um unico arquivo e fazer os testes de correlação e depois disso a regressão linear

In [135]:
df = pd.read_csv("analise_robusta.csv")

In [ ]:
abandono_em = df.melt(
    id_vars=["UF"],  
    value_vars=[col for col in df.columns if "Abandono_EM" in col],
    var_name="Ano", 
    value_name="Abandono_EM" 
)

abandono_em["Ano"] = abandono_em["Ano"].str.extract(r'(\d{4})').astype(int)
valor_bf = df.melt(
    id_vars=["UF"],
    value_vars=[col for col in df.columns if "valor_medio_bf_ajustado_" in col],
    var_name="Ano",
    value_name="Valor_medio_BF"
)
valor_bf["Ano"] = valor_bf["Ano"].str.extract(r'(\d{4})').astype(int)
df_final = pd.merge(abandono_em, valor_bf, on=["UF", "Ano"])
print(df_final.head())

   UF   Ano  Abandono_EM  Valor_medio_BF
0  AC  2007         5512          184.37
1  AL  2007        32874          175.03
2  AP  2007         8513          189.19
3  AM  2007        21077          189.42
4  BA  2007       161873          178.15


In [ ]:
Abando_ef = df.melt(
    id_vars=["UF"],
    value_vars=[col for col in df.columns if "Abandono_EF" in col],
    var_name="Ano",
    value_name="Abandono_EF"
)
Abando_ef["Ano"] = Abando_ef["Ano"].str.extract(r'(\d{4})').astype(int)



df_final = pd.merge(df_final, Abando_ef, on=["UF", "Ano"])
print(df_final.head())

   UF   Ano  Abandono_EM  Valor_medio_BF  Abandono_EF
0  AC  2007         5512          184.37         4470
1  AL  2007        32874          175.03        48121
2  AP  2007         8513          189.19         6930
3  AM  2007        21077          189.42        51244
4  BA  2007       161873          178.15       185146


In [ ]:
IDHM_EDUCACAO = df.melt(
    id_vars=["UF"],
    value_vars=[col for col in df.columns if "IDHM Educação" in col],
    var_name="Ano",
    value_name="IDHM Educação"
)
IDHM_EDUCACAO ["Ano"] = IDHM_EDUCACAO["Ano"].str.extract(r'(\d{4})').astype(int)
df_final = pd.merge(df_final, IDHM_EDUCACAO, on=["UF", "Ano"])

print(df_final.head())



   UF   Ano  Abandono_EM  Valor_medio_BF  Abandono_EF  IDHM Educação
0  AC  2012         4006          361.93         5270          0.649
1  AL  2012        23988          294.55        44245          0.587
2  AP  2012         6925          343.57         5039          0.659
3  AM  2012        21056          338.33        42742          0.632
4  BA  2012        83059          285.55       127753          0.593


In [ ]:
IDHM_RENDA = df.melt(
    id_vars=["UF"],
    value_vars=[col for col in df.columns if "IDHM Renda" in col],
    var_name="Ano",
    value_name="IDHM Renda"
)
IDHM_RENDA["Ano"] = IDHM_RENDA["Ano"].str.extract(r'(\d{4})').astype(int)

df_final = pd.merge(df_final, IDHM_RENDA, on=["UF", "Ano"])

print(df_final.head())


   UF   Ano  Abandono_EM  Valor_medio_BF  Abandono_EF  IDHM Educação  \
0  AC  2012         4006          361.93         5270          0.649   
1  AL  2012        23988          294.55        44245          0.587   
2  AP  2012         6925          343.57         5039          0.659   
3  AM  2012        21056          338.33        42742          0.632   
4  BA  2012        83059          285.55       127753          0.593   

   IDHM Renda  
0       0.670  
1       0.627  
2       0.673  
3       0.683  
4       0.666  


In [ ]:
INDICE_ATKINSON= df.melt(
    id_vars=["UF"],
    value_vars=[col for col in df.columns if "Índice de Atkinson - Renda" in col],
    var_name="Ano",
    value_name="Índice de Atkinson Renda"
)
INDICE_ATKINSON["Ano"] = INDICE_ATKINSON["Ano"].str.extract(r'(\d{4})').astype(int)
df_final = pd.merge(df_final, INDICE_ATKINSON, on=["UF", "Ano"])
print(df_final.head())

   UF   Ano  Abandono_EM  Valor_medio_BF  Abandono_EF  IDHM Educação  \
0  AC  2012         4006          361.93         5270          0.649   
1  AL  2012        23988          294.55        44245          0.587   
2  AP  2012         6925          343.57         5039          0.659   
3  AM  2012        21056          338.33        42742          0.632   
4  BA  2012        83059          285.55       127753          0.593   

   IDHM Renda  Índice de Atkinson Renda  
0       0.670                     0.422  
1       0.627                     0.352  
2       0.673                     0.361  
3       0.683                     0.417  
4       0.666                     0.405  


In [ ]:
INDICE_ATKINSON= df.melt(
    id_vars=["UF"],
    value_vars=[col for col in df.columns if "Índice de Atkinson - Educação" in col],
    var_name="Ano",
    value_name="Índice de Atkinson Educacao"
)
INDICE_ATKINSON["Ano"] = INDICE_ATKINSON["Ano"].str.extract(r'(\d{4})').astype(int)
df_final = pd.merge(df_final, INDICE_ATKINSON, on=["UF", "Ano"])
print(df_final.head())

   UF   Ano  Abandono_EM  Valor_medio_BF  Abandono_EF  IDHM Educação  \
0  AC  2012         4006          361.93         5270          0.649   
1  AL  2012        23988          294.55        44245          0.587   
2  AP  2012         6925          343.57         5039          0.659   
3  AM  2012        21056          338.33        42742          0.632   
4  BA  2012        83059          285.55       127753          0.593   

   IDHM Renda  Índice de Atkinson Renda  Índice de Atkinson Educacao  
0       0.670                     0.422                        0.281  
1       0.627                     0.352                        0.296  
2       0.673                     0.361                        0.184  
3       0.683                     0.417                        0.202  
4       0.666                     0.405                        0.275  


In [ ]:
QTD_FAMILIAS= df.melt(
    id_vars=["UF"],
    value_vars=[col for col in df.columns if "qtd_familias_beneficiarias_bolsa_familia_ajustado_" in col],
    var_name="Ano",
    value_name="QTD_FAMILIAS_BF"
)
QTD_FAMILIAS["Ano"] = QTD_FAMILIAS["Ano"].str.extract(r'(\d{4})').astype(int)
df_final = pd.merge(df_final,QTD_FAMILIAS, on=["UF", "Ano"])

print(df_final.head())

   UF   Ano  Abandono_EM  Valor_medio_BF  Abandono_EF  IDHM Educação  \
0  AC  2012         4006          361.93         5270          0.649   
1  AL  2012        23988          294.55        44245          0.587   
2  AP  2012         6925          343.57         5039          0.659   
3  AM  2012        21056          338.33        42742          0.632   
4  BA  2012        83059          285.55       127753          0.593   

   IDHM Renda  Índice de Atkinson Renda  Índice de Atkinson Educacao  \
0       0.670                     0.422                        0.281   
1       0.627                     0.352                        0.296   
2       0.673                     0.361                        0.184   
3       0.683                     0.417                        0.202   
4       0.666                     0.405                        0.275   

   QTD_FAMILIAS_BF  
0          67021.0  
1         429494.0  
2          52285.0  
3         323646.0  
4        1777116.0  


In [ ]:
gini=pd.read_csv("dadosUsadosParaMerge/idhm_gini_estado.csv")

IndiciDeGINI= gini.melt(
    id_vars=["UF"],
    value_vars=[col for col in gini.columns if "Índice de Gini " in col],
    var_name="Ano",
    value_name="IndiceDeGINI"
)
IndiciDeGINI["Ano"] = IndiciDeGINI["Ano"].str.extract(r'(\d{4})').astype(int)
df_final = pd.merge(df_final,IndiciDeGINI, on=["UF", "Ano"])
df_final.head()

,UF,Ano,Abandono_EM,Valor_medio_BF,Abandono_EF,IDHM Educação,IDHM Renda,Índice de Atkinson Renda,Índice de Atkinson Educacao,QTD_FAMILIAS_BF,rendaPercapta,IndiceDeGINI
0,AC,2012,4006,361.93,5270,0.649,0.670,0.422,0.281,67021.0,516.75,0.566
1,AL,2012,23988,294.55,44245,0.587,0.627,0.352,0.296,429494.0,395.06,0.503
2,AP,2012,6925,343.57,5039,0.659,0.673,0.361,0.184,52285.0,528.23,0.528
3,AM,2012,21056,338.33,42742,0.632,0.683,0.417,0.202,323646.0,559.44,0.589
4,BA,2012,83059,285.55,127753,0.593,0.666,0.405,0.275,1777116.0,503.29,0.563


In [ ]:
taxas = pd.read_csv("dadosUsadosParaMerge/TaxasDeDesocupacaoOcupacaoParticipacao.csv")

TAXADESOCUPACAO = taxas[["UF","Ano","Taxa_Desocupacao"]]
df_final = pd.merge(df_final,TAXADESOCUPACAO, on=["UF", "Ano"])
df_final.head()

TAXADEOCUPACAO = taxas[["UF","Ano","Nivel_Ocupacao"]]
df_final = pd.merge(df_final,TAXADEOCUPACAO, on=["UF", "Ano"])
df_final.head()

TAXAPARTICIPACAO = taxas[["UF","Ano","Taxa_Participacao"]]
df_final = pd.merge(df_final,TAXAPARTICIPACAO, on=["UF", "Ano"])
df_final.head()

,UF,Ano,Abandono_EM,Valor_medio_BF,Abandono_EF,IDHM Educação,IDHM Renda,Índice de Atkinson Renda,Índice de Atkinson Educacao,QTD_FAMILIAS_BF,rendaPercapta,IndiceDeGINI,Taxa_Desocupacao
0,AC,2012,4006,361.93,5270,0.649,0.67,0.422,0.281,67021.0,516.75,0.566,9.3
1,AC,2012,4006,361.93,5270,0.649,0.67,0.422,0.281,67021.0,516.75,0.566,9.1
2,AC,2012,4006,361.93,5270,0.649,0.67,0.422,0.281,67021.0,516.75,0.566,7.7
3,AC,2012,4006,361.93,5270,0.649,0.67,0.422,0.281,67021.0,516.75,0.566,8.1
4,AC,2012,4006,361.93,5270,0.649,0.67,0.422,0.281,67021.0,516.75,0.566,10.9
